In [ ]:
# 녹음된 음성 데이터를 전처리하는 과정입니다.
# 녹음된 음성데이터는 비트메이트_TP01/data/dataset/wav_{speaker}/raw폴더에 있어야합니다.
# raw 폴더 내 음성데이터를 변환해서 wavs 폴더에 저장합니다.

# .m4a .wav 상관 없이 .wav 로 변환해줍니다.
# samplingrate는 24000으로 변경합니다. (finetunning시 24000으로 학습시켜야함)
# 무음 구간은 제거합니다
# 볼륨도 일정하게 조절합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import numpy as np
import librosa
import soundfile as sf

# =========================
# 경로 설정
# =========================
SPEAKER = "jhc100"

BASE_DIR = Path("/content/drive/MyDrive/비트메이트_TP01")
DATASET_DIR = BASE_DIR / "data" / "dataset" / f"wav_{SPEAKER}"

INPUT_DIR = DATASET_DIR / "raw"
OUTPUT_DIR = DATASET_DIR / "wavs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 24000

# =========================
# 파일 리스트
# =========================
audio_files = [
    p for p in INPUT_DIR.iterdir()
    if p.suffix.lower() in [".wav", ".m4a"]
]

if not audio_files:
    raise RuntimeError(f"오디오 파일 없음: {INPUT_DIR}")

print(f"총 파일 수: {len(audio_files)}")

# =========================
# 전처리
# =========================
for input_path in audio_files:
    output_path = OUTPUT_DIR / (input_path.stem + ".wav")

    try:
        # 1. 로드 + mono + resample
        audio, sr = librosa.load(
            str(input_path),
            sr=TARGET_SR,
            mono=True
        )

        # 2. 무음 제거 (앞뒤)
        audio, _ = librosa.effects.trim(
            audio,
            top_db=30
        )

        # 3. normalize (볼륨 일정)
        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val

        # 4. 너무 짧은 음성 제거 (노이즈 방지)
        if len(audio) < TARGET_SR * 0.3:  # 0.3초 이하 컷
            print(f"[스킵] 너무 짧음: {input_path.name}")
            continue

        # 5. 저장
        sf.write(str(output_path), audio, TARGET_SR)

        print(f"[완료] {input_path.name} → {output_path.name}")

    except Exception as e:
        print(f"[에러] {input_path.name}: {e}")